<a href="https://colab.research.google.com/github/enm0910/ST554/blob/main/EMartinez_Project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Author: Emma Martinez

Course: ST 554 Project 3

Purpose: Project 3- PySpark ML + Streaming Project

# Project #3

This notebook has two parts:

**Part 1** – Fits an elastic net regression model to predict Power_Zone_3 (power consumption in Tetouan City) using PySpark's MLlib. We load the data, build a transformation pipeline (binarization, one-hot encoding, PCA), and tune the model via 5-fold cross-validation across a grid of regularization parameters.

**Part 2** – Reads a live stream of incoming CSV files using PySpark Structured Streaming. We apply the fitted model to each batch of arriving data to generate real-time predictions, compute residuals, and write the results to the console.

We start every section by installing PySpark and creating (or reusing) a SparkSession.

### Setup

In [15]:
# Install PySpark for every session
!pip install pyspark

In [16]:
# Imports
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField,
                                DoubleType, IntegerType, StringType)
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Binarizer, OneHotEncoder, StringIndexer,
    VectorAssembler, PCA, SQLTransformer
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

In [17]:
# Start a local Spark session
spark = SparkSession.builder \
    .appName("PowerZone3Prediction") \
    .getOrCreate()

# Surpress normal Spark logs
spark.sparkContext.setLogLevel("ERROR")
print("Spark session started")

Spark session started


### Part 1: Model Fitting

#### 1.1: Load data

In [18]:
# Read csv into pandas and convert to Spark
pandas_df = pd.read_csv("/content/power_ml_data.csv")

# Cast Month to string so StringIndexer works right
pandas_df["Month"] = pandas_df["Month"].astype(str)

spark_df = spark.createDataFrame(pandas_df)

spark_df.printSchema()
spark_df.show(5)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: string (nullable = true)
 |-- Hour: long (nullable = true)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375

####1.2: Define pipline stages

In [19]:
# Stage 1: Cast Hour to doubletype (for binarizer)
sql_transform = SQLTransformer(
    statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_dbl FROM __THIS__"
)

# Stage 2: Binarize Hour- 0 if night (< 6.5), 1 if day (>= 6.5)
binarizer = Binarizer(
    threshold=6.5,
    inputCol="Hour_dbl",
    outputCol="Hour_binary"
)

# Stage 3 & 4: Index then one-hot encode Month
month_indexer = StringIndexer(inputCol="Month", outputCol="Month_index")
month_encoder = OneHotEncoder(inputCols=["Month_index"], outputCols=["Month_ohe"])

# Stage 5: Assemble weather columns into a vector for PCA
weather_cols = ["Temperature", "Humidity", "Wind_Speed",
                "General_Diffuse_Flows", "Diffuse_Flows"]

weather_assembler = VectorAssembler(
    inputCols=weather_cols,
    outputCol="weather_features"
)

# Stage 6: PCA— reduce weather columns to 2 principal components
pca = PCA(k=2, inputCol="weather_features", outputCol="pca_features")

# Stage 7: Rename Power_Zone_3; label (for Spark ML estimators)
label_transform = SQLTransformer(
    statement="SELECT *, Power_Zone_3 AS label FROM __THIS__"
)

# Stage 8: Assemble final feature vector
final_assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_binary",
               "Power_Zone_1", "Power_Zone_2", "Month_ohe"],
    outputCol="features"
)

####1.3: Build pipeline

In [20]:
# Chain all stages into a single Pipeline
pipeline = Pipeline(stages=[
    sql_transform,
    binarizer,
    month_indexer,
    month_encoder,
    weather_assembler,
    pca,
    label_transform,
    final_assembler
])

####1.4 Set up cross-validation

In [21]:
# Elastic net linear regression
lr = LinearRegression(featuresCol="features", labelCol="label")

# All 121 combos of these values will be tried
param_values = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]

param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, param_values) \
    .addGrid(lr.elasticNetParam, param_values) \
    .build()

# RMSE as error metric
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

# 5-fold cross-validation over full pipeline + LR model
cv = CrossValidator(
    estimator=Pipeline(stages=pipeline.getStages() + [lr]),
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

####1.5: Fit the model

In [22]:
# Fit the model
print("Fitting model")
cv_model = cv.fit(spark_df)
print("Done")

Fitting model
Done
